In [3]:
import kagglehub
from pathlib import Path

folderpath = Path(kagglehub.dataset_download("kagglevladkh/imagenet1k-animals-full-split"))

print("Path to dataset files:", folderpath, type(folderpath))

100%|██████████| 20.3G/20.3G [02:50<00:00, 128MB/s] 

Extracting files...


Path to dataset files: /teamspace/studios/this_studio/.cache/kagglehub/datasets/kagglevladkh/imagenet1k-animals-full-split/versions/1 <class 'pathlib.PosixPath'>


In [4]:
train_dir = folderpath / "train"
test_dir = folderpath / "test"

print(train_dir, test_dir)

/teamspace/studios/this_studio/.cache/kagglehub/datasets/kagglevladkh/imagenet1k-animals-full-split/versions/1/train /teamspace/studios/this_studio/.cache/kagglehub/datasets/kagglevladkh/imagenet1k-animals-full-split/versions/1/test


In [5]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode


normalize = transforms.Normalize(
	mean=[0.485, 0.456, 0.406],
	std=[0.229, 0.224, 0.225]
)

train_transform = transforms.Compose([
	transforms.RandomResizedCrop(
		size=(128, 128),
		scale=(0.75, 1.0),
		ratio=(0.8, 1.25),
		interpolation=InterpolationMode.BICUBIC
	),
	transforms.RandomHorizontalFlip(p=0.5),
	transforms.RandomRotation(degrees=10),
	transforms.ColorJitter(
		brightness=0.15,
		contrast=0.15,
		saturation=0.15
	),
	transforms.ToTensor(),
	normalize
])

test_transform = transforms.Compose([
	transforms.Resize(
		size=144,
		interpolation=InterpolationMode.BICUBIC
	),
	transforms.CenterCrop(size=(128, 128)),
	transforms.ToTensor(),
	normalize
])

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='cuda')

In [7]:
from mlutils.data_setup import create_folder_dataloaders

train_dataloader, test_dataloader, class_names, class_to_idx = create_folder_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    train_transform=train_transform,
    test_transform=test_transform,
    batch_size=32
)

In [8]:
print(len(train_dataloader), len(test_dataloader), len(class_names), len(class_to_idx))

13094 3279 396 396


In [9]:
num_classes = len(class_names)

In [10]:
class ConvBNAct(nn.Sequential):
	def __init__(
		self,
		in_channels,
		out_channels,
		kernel_size,
		stride=1,
		groups=1
	):
		padding = kernel_size // 2

		super().__init__(
			nn.Conv2d(
				in_channels=in_channels,
				out_channels=out_channels,
				kernel_size=kernel_size,
				stride=stride,
				padding=padding,
				groups=groups,
				bias=False
			),
			nn.BatchNorm2d(
				out_channels,
				eps=0.001,
				momentum=0.01
			),
			nn.SiLU(inplace=True)
		)

In [11]:
class SqueezeExcitation(nn.Module):
	def __init__(self, input_channels, squeeze_channels):
		super().__init__()

		self.avgpool = nn.AdaptiveAvgPool2d(1)

		self.fc1 = nn.Conv2d(
			in_channels=input_channels,
			out_channels=squeeze_channels,
			kernel_size=1
		)

		self.act = nn.SiLU(inplace=True)

		self.fc2 = nn.Conv2d(
			in_channels=squeeze_channels,
			out_channels=input_channels,
			kernel_size=1
		)

		self.scale_act = nn.Sigmoid()

	def forward(self, x):
		scale = self.avgpool(x)
		scale = self.fc1(scale)
		scale = self.act(scale)
		scale = self.fc2(scale)
		scale = self.scale_act(scale)

		return x * scale

In [12]:
class StochasticDepth(nn.Module):
	def __init__(self, p):
		super().__init__()
		self.p = p

	def forward(self, x):
		if not self.training or self.p == 0:
			return x

		survival_rate = 1.0 - self.p

		shape = [x.shape[0]] + [1] * (x.ndim - 1)

		random_tensor = survival_rate + torch.rand(
			shape,
			dtype=x.dtype,
			device=x.device
		)

		binary_tensor = torch.floor(random_tensor)

		return x / survival_rate * binary_tensor

In [13]:
class EfficientNetMBConv(nn.Module):
	def __init__(
		self,
		input_channels,
		output_channels,
		expand_ratio,
		kernel_size,
		stride,
		stochastic_depth_prob=0.0
	):
		super().__init__()

		self.use_residual = (
			stride == 1 and input_channels == output_channels
		)

		expanded_channels = input_channels * expand_ratio

		layers = []

		# 1. Expansion: 1x1 Conv + BN + SiLU
		if expand_ratio != 1:
			layers.append(
				ConvBNAct(
					in_channels=input_channels,
					out_channels=expanded_channels,
					kernel_size=1
				)
			)

		# 2. Depthwise: 3x3 або 5x5 Conv + BN + SiLU
		layers.append(
			ConvBNAct(
				in_channels=expanded_channels,
				out_channels=expanded_channels,
				kernel_size=kernel_size,
				stride=stride,
				groups=expanded_channels
			)
		)

		# 3. Squeeze-and-Excitation
		squeeze_channels = max(1, input_channels // 4)

		layers.append(
			SqueezeExcitation(
				input_channels=expanded_channels,
				squeeze_channels=squeeze_channels
			)
		)

		# 4. Projection: linear 1x1 Conv + BN
		layers.append(
			nn.Conv2d(
				in_channels=expanded_channels,
				out_channels=output_channels,
				kernel_size=1,
				bias=False
			)
		)

		layers.append(
			nn.BatchNorm2d(
				output_channels,
				eps=0.001,
				momentum=0.01
			)
		)

		self.block = nn.Sequential(*layers)

		self.stochastic_depth = StochasticDepth(stochastic_depth_prob)

	def forward(self, x):
		result = self.block(x)

		if self.use_residual:
			result = self.stochastic_depth(result)
			result = result + x

		return result

In [14]:
class ReplicaEffNetB0(nn.Module):
  def __init__(
    self,
    num_classes,
    dropout=0.2,
    stochastic_depth_prob=0.2
  ):
    super().__init__()

    self.features = nn.Sequential(
      # Stage 1
      # Stem, k3x3, 3 -> 32, stride=2, layers=1
      ConvBNAct(
        in_channels=3,
        out_channels=32,
        kernel_size=3,
        stride=2
      ),

      # Stage 2
      # MBConv1, k3x3, 32 -> 16, stride=1, layers=1
      EfficientNetMBConv(
        input_channels=32,
        output_channels=16,
        expand_ratio=1,
        kernel_size=3,
        stride=1,
        stochastic_depth_prob=0.0
      ),

      # Stage 3
      # MBConv6, k3x3, 16 -> 24, stride=2, layers=2
      EfficientNetMBConv(
        input_channels=16,
        output_channels=24,
        expand_ratio=6,
        kernel_size=3,
        stride=2,
        stochastic_depth_prob=0.0125
      ),
      EfficientNetMBConv(
        input_channels=24,
        output_channels=24,
        expand_ratio=6,
        kernel_size=3,
        stride=1,
        stochastic_depth_prob=0.025
      ),

      # Stage 4
      # MBConv6, k5x5, 24 -> 40, stride=2, layers=2
      EfficientNetMBConv(
        input_channels=24,
        output_channels=40,
        expand_ratio=6,
        kernel_size=5,
        stride=2,
        stochastic_depth_prob=0.0375
      ),
      EfficientNetMBConv(
        input_channels=40,
        output_channels=40,
        expand_ratio=6,
        kernel_size=5,
        stride=1,
        stochastic_depth_prob=0.05
      ),

      # Stage 5
      # MBConv6, k3x3, 40 -> 80, stride=2, layers=3
      EfficientNetMBConv(
        input_channels=40,
        output_channels=80,
        expand_ratio=6,
        kernel_size=3,
        stride=2,
        stochastic_depth_prob=0.0625
      ),
      EfficientNetMBConv(
        input_channels=80,
        output_channels=80,
        expand_ratio=6,
        kernel_size=3,
        stride=1,
        stochastic_depth_prob=0.075
      ),
      EfficientNetMBConv(
        input_channels=80,
        output_channels=80,
        expand_ratio=6,
        kernel_size=3,
        stride=1,
        stochastic_depth_prob=0.0875
      ),

      # Stage 6
      # MBConv6, k5x5, 80 -> 112, stride=1, layers=3
      EfficientNetMBConv(
        input_channels=80,
        output_channels=112,
        expand_ratio=6,
        kernel_size=5,
        stride=1,
        stochastic_depth_prob=0.1
      ),
      EfficientNetMBConv(
        input_channels=112,
        output_channels=112,
        expand_ratio=6,
        kernel_size=5,
        stride=1,
        stochastic_depth_prob=0.1125
      ),
      EfficientNetMBConv(
        input_channels=112,
        output_channels=112,
        expand_ratio=6,
        kernel_size=5,
        stride=1,
        stochastic_depth_prob=0.125
      ),

      # Stage 7
      # MBConv6, k5x5, 112 -> 192, stride=2, layers=4
      EfficientNetMBConv(
        input_channels=112,
        output_channels=192,
        expand_ratio=6,
        kernel_size=5,
        stride=2,
        stochastic_depth_prob=0.1375
      ),
      EfficientNetMBConv(
        input_channels=192,
        output_channels=192,
        expand_ratio=6,
        kernel_size=5,
        stride=1,
        stochastic_depth_prob=0.15
      ),
      EfficientNetMBConv(
        input_channels=192,
        output_channels=192,
        expand_ratio=6,
        kernel_size=5,
        stride=1,
        stochastic_depth_prob=0.1625
      ),
      EfficientNetMBConv(
        input_channels=192,
        output_channels=192,
        expand_ratio=6,
        kernel_size=5,
        stride=1,
        stochastic_depth_prob=0.175
      ),

      # Stage 8
      # MBConv6, k3x3, 192 -> 320, stride=1, layers=1
      EfficientNetMBConv(
        input_channels=192,
        output_channels=320,
        expand_ratio=6,
        kernel_size=3,
        stride=1,
        stochastic_depth_prob=0.1875
      ),

      # Stage 9
      # Head Conv 1x1 & pooling & FC
			ConvBNAct(
				in_channels=320,
				out_channels=1280,
				kernel_size=1,
				stride=1
			)
    )

    self.avgpool = nn.AdaptiveAvgPool2d(1)

    self.classifier = nn.Sequential(
      nn.Dropout(p=dropout, inplace=True),
      nn.Linear(1280, num_classes)
		)

  def forward(self, x):
    x = self.features(x)
    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    x = self.classifier(x)

    return x

In [15]:
model_0 = ReplicaEffNetB0(num_classes=num_classes)

In [16]:
from mlutils.engine import train

EPOCHS = 29

loss_fn = nn.CrossEntropyLoss(
	label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
	model_0.parameters(),
	lr=3e-4,
	weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
	optimizer,
	T_max=EPOCHS,
	eta_min=1e-6
)

model_0_results = train(
	model=model_0,
	train_dataloader=train_dataloader,
	test_dataloader=test_dataloader,
	optimizer=optimizer,
	loss_fn=loss_fn,
	epochs=EPOCHS,
	device=device,
	scheduler=scheduler,
	top_k=5,
	use_amp=True,
	grad_clip=1.0,
	save_best_path=Path("full_best_effnetb0_replica.pth"),
	save_last_path=Path("full_last_effnetb0_replica.pth"),
	save_best_by="test_acc"
)

  0%|          | 0/60 [00:00<?, ?it/s]

Epoch: 1 | lr: 0.00029980 | train_loss: 4.8193 | train_acc: 0.0855 | test_loss: 4.0765 | test_acc: 0.1733 | test_top5_acc: 0.4063
Epoch: 2 | lr: 0.00029918 | train_loss: 3.7827 | train_acc: 0.2286 | test_loss: 3.3565 | test_acc: 0.3046 | test_top5_acc: 0.5789
Epoch: 3 | lr: 0.00029816 | train_loss: 3.2376 | train_acc: 0.3309 | test_loss: 2.9421 | test_acc: 0.3897 | test_top5_acc: 0.6692
Epoch: 4 | lr: 0.00029673 | train_loss: 2.8867 | train_acc: 0.4032 | test_loss: 2.6813 | test_acc: 0.4467 | test_top5_acc: 0.7208
Epoch: 5 | lr: 0.00029491 | train_loss: 2.6435 | train_acc: 0.4555 | test_loss: 2.4506 | test_acc: 0.4987 | test_top5_acc: 0.7672
Epoch: 6 | lr: 0.00029268 | train_loss: 2.4628 | train_acc: 0.4958 | test_loss: 2.3273 | test_acc: 0.5253 | test_top5_acc: 0.7902
Epoch: 7 | lr: 0.00029007 | train_loss: 2.3230 | train_acc: 0.5279 | test_loss: 2.2120 | test_acc: 0.5547 | test_top5_acc: 0.8104
Epoch: 8 | lr: 0.00028708 | train_loss: 2.2097 | train_acc: 0.5549 | test_loss: 2.1608 | t